# Fase 1 · SAMPLE — Muestreo y selección de datos

**Objetivo:** construir un conjunto de datos representativo con ejemplos de código fuente y de
configuración, vulnerables y no vulnerables, asociados a las categorías **OWASP Top 10:2025** y
**OWASP API Security Top 10:2023**.

El dataset se construye con `scripts/construir_dataset.py`. Cada registro es una **unidad**: una
función, un *handler* de una ruta o un bloque de configuración (un servicio de `docker-compose`, una
sección de `application.yml`, una etapa de un `Dockerfile`, una ruta de OpenAPI). Uso exactamente el
mismo segmentador que la herramienta CLI, de modo que el modelo aprende con el mismo tipo de
fragmento que luego clasifica en un proyecto real.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
sys.path.insert(0, str(BASE / "notebooks"))
pd.set_option("display.max_colwidth", 80)
df = pd.read_csv(BASE / "datos" / "dataset.csv", keep_default_na=False)
print(f"{len(df):,} unidades etiquetadas | {df.vulnerable.sum():,} vulnerables | {(df.vulnerable == 0).sum():,} no vulnerables")

14,329 unidades etiquetadas | 5,134 vulnerables | 9,195 no vulnerables


## 1. Fuentes de datos

| Fuente | Qué aporta | Cómo se obtiene la etiqueta |
|---|---|---|
| **semgrep-rules** | Pruebas de reglas de seguridad (Java, Python, JS, TS, Dockerfile, docker-compose) | Línea anotada `ruleid:` (vulnerable) u `ok:` (seguro); CWE en los metadatos de la regla |
| **CodeQL** (GitHub) | Pruebas de consultas de seguridad | Comentarios `$ Alert`, `BAD`, `NOT OK` / `GOOD`, `OK`; CWE en el nombre del directorio |
| **OWASP Benchmark** Java v1.2 y Python v0.1 | Casos de prueba con resultado esperado (fuente sugerida en el enunciado) | Archivo `expectedresults` (verdadero/falso) y su CWE |
| **KICS** (Checkmarx) | Pruebas positivas y negativas de Dockerfile, docker-compose y OpenAPI | Línea esperada de la prueba positiva; consultas de seguridad seleccionadas |
| **SecurityEval** | Ejemplos Python inseguros clasificados por CWE | Etiqueta del conjunto |
| **Casos controlados** | Pares vulnerable/corregido de API (BOLA, BFLA, asignación masiva, consumo de recursos, inventario...) en Express, NestJS, FastAPI, Flask y Spring Boot | Etiqueta por diseño (fuente sugerida en el enunciado) |
| **Código benigno** | 25 proyectos reales (boilerplates de NestJS/Express, Spring Boot, FastAPI, Flask, Django...) | Código de producción sin vulnerabilidades conocidas |

La versión exacta (commit) de cada repositorio queda registrada en `datos/versiones_fuentes.json`.

In [2]:
versiones = json.loads((BASE / "datos" / "versiones_fuentes.json").read_text())
pd.DataFrame([{"nombre": k, "repositorio": v["repositorio"], "commit": v["commit"][:10], "uso": v["uso"]}
              for k, v in versiones.items()]).sort_values(["uso", "nombre"]).reset_index(drop=True)

,nombre,repositorio,commit,uso
0,NodeGoat,https://github.com/OWASP/NodeGoat,c5cb68a708,desarrollo
1,ftgo-application,https://github.com/microservices-patterns/ftgo-application,558dfc53b1,desarrollo
2,pygoat,https://github.com/adeyosemanputra/pygoat,2aa4cc14f2,desarrollo
3,robot-shop,https://github.com/instana/robot-shop,55292e2199,desarrollo
4,benchmark-java,https://github.com/OWASP-Benchmark/BenchmarkJava,10dfcc54ad,entrenamiento
5,benchmark-python,https://github.com/OWASP-Benchmark/BenchmarkPython,f129148580,entrenamiento
6,benigno/awesome-compose,https://github.com/docker/awesome-compose,503c0dcef3,entrenamiento
7,benigno/awesome-nest-boilerplate,https://github.com/NarHakobyan/awesome-nest-boilerplate,28bc9bb294,entrenamiento
8,benigno/dispatch,https://github.com/Netflix/dispatch,dd2837e82a,entrenamiento
9,benigno/django-realworld,https://github.com/gothinkster/django-realworld-example-app,29c9d42831,entrenamiento


In [3]:
tabla = (df.groupby("fuente").agg(unidades=("id", "size"), vulnerables=("vulnerable", "sum"),
                                  proyectos=("proyecto", "nunique"))
         .assign(no_vulnerables=lambda t: t.unidades - t.vulnerables)
         .sort_values("unidades", ascending=False))
tabla.loc["Total"] = tabla.sum()
tabla

,unidades,vulnerables,proyectos,no_vulnerables
fuente,,,,
benigno,6261,0,62,6261
codeql-tests,2188,1820,232,368
semgrep-rules,2061,1384,120,677
casos-controlados,1288,647,103,641
owasp-benchmark-java,1188,591,888,597
owasp-benchmark-python,609,225,609,384
kics,348,154,57,194
vulnerableapp,172,123,22,49
securityeval,118,118,102,0


## 2. Diccionario de datos

In [4]:
pd.read_csv(BASE / "datos" / "diccionario_datos.csv")

,columna,tipo,descripcion,ejemplo
0,id,texto,Identificador único de la unidad en el dataset,u00042
1,fuente,categórica,"Origen del registro: semgrep-rules, codeql-tests, owasp-benchmark-java, owas...",codeql-tests
2,proyecto,texto,"Grupo de origen (proyecto, archivo de prueba o familia de casos). Todas las ...",codeql:java/ql/test/query-tests/security/CWE-089
3,archivo,texto,Ruta del archivo de origen dentro de la fuente,src/main/java/.../BenchmarkTest00008.java
4,lenguaje,categórica,"Lenguaje de la unidad: java, python, javascript, typescript o config (archiv...",java
5,subtipo,categórica,"Tipo de unidad: codigo, dockerfile, compose, yaml, properties, env, openapi ...",codigo
6,unidad,texto,"Nombre de la función, handler o sección de configuración",doPost
7,linea_inicio,entero,Primera línea de la unidad en el archivo de origen,24
8,linea_fin,entero,Última línea de la unidad en el archivo de origen,51
9,codigo,texto,Texto de la unidad tal como aparece en el archivo,"@GetMapping(""/{id}"") public Order get(...)"


Ejemplo de un registro (una unidad vulnerable de un caso controlado):

In [5]:
ejemplo = df[(df.fuente == "casos-controlados") & (df.api2023.str.contains("API1"))].iloc[0]
print(ejemplo[["fuente", "archivo", "lenguaje", "unidad", "cwe", "owasp2025", "api2023", "origen_etiqueta"]].to_string())
print("\n" + ejemplo.codigo)

fuente                                 casos-controlados
archivo                         bola/drf_c_20_1/views.py
lenguaje                                          python
unidad                                               get
cwe                                              CWE-639
owasp2025                                       A01:2025
api2023                                        API1:2023
origen_etiqueta    caso controlado (etiqueta por diseño)

    def get(self, request, pk=None):
        logger.debug("detail requested for document %s", pk)
        try:
            document = Document.objects.select_related("customer").get(pk=pk)
        except Document.DoesNotExist:
            return Response({"detail": "not found"}, status=status.HTTP_404_NOT_FOUND)
        serializer = DocumentSerializer(document)
        return Response(serializer.data, status=status.HTTP_200_OK)


## 3. Correspondencia CWE – OWASP

La tabla se genera con `scripts/construir_mapeo_cwe.py` a partir de los **documentos oficiales**:
la sección *List of Mapped CWEs* de cada categoría del OWASP Top 10:2025 y las referencias externas
de cada riesgo del OWASP API Security Top 10:2023. Algunos CWE no figuran en las listas oficiales,
pero su CWE padre en la jerarquía de MITRE sí (por ejemplo, CWE-943 *NoSQL Injection*, hijo de
CWE-74). Esas relaciones se marcan como **extendidas** y llevan su justificación.

In [6]:
corr = pd.read_csv(BASE / "datos" / "correspondencias_cwe_owasp.csv", keep_default_na=False)
print(f"{len(corr)} CWE en la tabla | Top 10 oficial: {(corr.tipo_top10 == 'oficial').sum()} | "
      f"Top 10 extendida: {(corr.tipo_top10 == 'extendida').sum()} | con categoría API: {(corr.owasp_api_2023 != '').sum()}")
corr[corr.owasp_api_2023 != ""][["cwe", "nombre", "owasp_top10_2025", "owasp_api_2023", "tipo_api"]]

264 CWE en la tabla | Top 10 oficial: 249 | Top 10 extendida: 15 | con categoría API: 50


,cwe,nombre,owasp_top10_2025,owasp_api_2023,tipo_api
0,CWE-2,,A02:2025,API8:2023,oficial
5,CWE-16,,A02:2025,API8:2023,oficial
6,CWE-20,Improper Input Validation,A05:2025,API10:2023,oficial
44,CWE-200,Exposure of Sensitive Information to an Unauthorized Actor,A01:2025,API10:2023,oficial
46,CWE-204,Observable Response Discrepancy,A07:2025,API2:2023,oficial
47,CWE-209,Generation of Error Message Containing Sensitive Information,A10:2025,API8:2023,oficial
48,CWE-213,Exposure of Sensitive Information Due to Incompatible Policies,A01:2025,API3:2023,oficial
49,CWE-215,Insertion of Sensitive Information Into Debugging Code,A10:2025,API8:2023,extendida
56,CWE-250,Execution with Unnecessary Privileges,A02:2025,API8:2023,extendida
58,CWE-256,Plaintext Storage of a Password,A06:2025,API2:2023,extendida


## 4. Justificación del procedimiento de muestreo

- **Unidades de análisis realistas.** Se etiqueta la función o bloque completo que contiene la línea
  anotada, con el mismo segmentador que usa la herramienta.
- **Pares vulnerable/corregido.** Semgrep, CodeQL, KICS y los casos controlados aportan versiones
  corregidas casi idénticas a las vulnerables: obligan al modelo a fijarse en la diferencia relevante
  (parámetros enlazados, verificación del propietario, rol, límites...) y no en palabras sueltas.
- **Código normal de proyectos reales.** Sin él, el modelo marcaría cualquier código que no se parezca
  a un ejemplo de libro. Se tomaron hasta 600 unidades por proyecto de 25 proyectos y se excluyeron las
  carpetas de ejemplos y pruebas, que a menudo contienen código inseguro a propósito.
- **Control de fugas de etiqueta.** Se eliminan los comentarios que delatan la clase (`ruleid`, `BAD`,
  `vulnerable`...) y se neutralizan los identificadores como `bad1()` o `BenchmarkTest00042`.
- **Balance.** Como máximo 12 unidades por archivo y 420 por CWE y fuente, para que ningún origen domine.
- **Limpieza.** Se descartan duplicados, unidades con etiquetas contradictorias y vulnerabilidades sin
  categoría OWASP (p. ej. seguridad de memoria).

## 5. Partición entrenamiento / validación / prueba (70/15/15)

La partición se hace **por grupo de origen**: todas las unidades de un mismo proyecto, del mismo
directorio de pruebas de una regla o de la misma familia de casos controlados (framework + estilo)
quedan en la misma partición. Así ninguna partición comparte procedencia con otra y la prueba mide la
generalización a proyectos y estilos no vistos. Los grupos se asignan con **estratificación iterativa
multietiqueta** (Sechidis et al., 2011), de modo que cada categoría OWASP aparezca en las tres particiones.

In [7]:
part = (df.groupby("particion").agg(unidades=("id", "size"), vulnerables=("vulnerable", "sum"),
                                    grupos=("proyecto", "nunique"))
        .reindex(["entrenamiento", "validacion", "prueba"]))
part["porcentaje"] = (100 * part.unidades / part.unidades.sum()).round(1)
part

,unidades,vulnerables,grupos,porcentaje
particion,,,,
entrenamiento,9927,3527,1541,69.3
validacion,2202,786,413,15.4
prueba,2200,821,263,15.4


In [8]:
# Comprobación: ningún grupo de origen ni ningún fragmento de código aparece en dos particiones
from vulnminer.lexico import limpiar_codigo
import re

assert (df.groupby("proyecto").particion.nunique() == 1).all()
norm = df.codigo.map(lambda c: re.sub(r"\s+", " ", limpiar_codigo(c)).strip())
assert norm.is_unique
print("Sin fugas entre particiones: OK")

Sin fugas entre particiones: OK


In [9]:
vul = df[df.vulnerable == 1]
filas = []
for col in ("owasp2025", "api2023"):
    e = vul.assign(c=vul[col].str.split(";")).explode("c").reset_index(drop=True)
    filas.append(pd.crosstab(e[e.c != ""].c, e[e.c != ""].particion))
cobertura = pd.concat(filas)[["entrenamiento", "validacion", "prueba"]]
cobertura

particion,entrenamiento,validacion,prueba
c,,,
A01:2025,709,159,152
A02:2025,263,60,58
A03:2025,33,15,13
A04:2025,500,108,117
A05:2025,1335,286,289
A06:2025,260,57,56
A07:2025,209,46,45
A08:2025,154,35,77
A09:2025,28,11,6


**Conclusión de la fase.** El dataset tiene muchos más registros que el mínimo exigido (500, con 100 o
más no vulnerables), proviene de fuentes públicas trazables más casos controlados propios, cubre los
cuatro lenguajes del alcance y los archivos de configuración, y está dividido sin fugas entre particiones.
La categoría **API6:2023** (flujos de negocio sensibles) no tiene ejemplos, porque depende de la lógica
de negocio de cada aplicación, y **API9:2023** tiene muy pocos: ambas se tratan en las fases siguientes.